In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import re
!pip install stanza
import stanza
stanza.download('ar')
import requests

#NLP Lib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.isri import ISRIStemmer


#Word Embidding lib 
from gensim.models import Word2Vec

#Vectorization & ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 
from bs4 import BeautifulSoup
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns



nltk.download('punkt')
nltk.download('stopwords')



In [ ]:
#preprocessing & pipeline
#1.Loading Data

file_path="/kaggle/input/datasets/ahmeddulaimi/ultimate-arabic-news-dataset/Original_Without_Stop_Stem.csv"
df=pd.read_csv(file_path)
print (df.head())

In [ ]:
#Text cleaning 

def basic_cleaning(text):
    text = re.sub(r'http\S+|www\S+|https\S+','',str(text),flags=re.MULTILINE)
    text = re.sub(r'\d+','',text)
    text=re.sub(r'[^\w\s]','',text)
    #Normaliation
    text=re.sub("[اإآأ]","ا",text)
    text=re.sub("ى","ي",text)
    text=re.sub("ة","ه",text)
    text=re.sub("ؤ","و",text)
    text=re.sub("ئ","ي",text)
    #Tashkeel:
    tashkeel_pattern=re.compile(r'[\u064B-\u0652]') 
    text=re.sub(tashkeel_pattern,'',text) 
    return text.strip()

df['cleaned_text'] = df['text'].apply(basic_cleaning)
print(df['text'].head())

In [ ]:
#Tokenization & Stop-words
arabic_stop_words = set(stopwords.words('arabic'))
def tokenize_and_remove_stop(text):
    tokens = word_tokenize(text)
    filtered_tokens = [w for w in tokens if not w in arabic_stop_words]
    return filtered_tokens

df['tokens']=df['cleaned_text'].apply(tokenize_and_remove_stop)
print(df[['cleaned_text','tokens']].head())

In [ ]:
#Stemming

st = ISRIStemmer()

def apply_stemming(tokens):
    stemmed_tokens = [st.stem(w) for w in tokens]
    return "  ".join(stemmed_tokens)

df['stemmed_text'] =df['tokens'].apply(apply_stemming)
print(df['stemmed_text'].head())

In [ ]:
#Stanza Pipeline


nlp = stanza.Pipeline('ar',processors = 'tokenize,pos,lemma,ner')


def extract_linguistic_features(text):
    doc = nlp(text)
    pos=[f"{w.text}({w.upos})"for s in doc.sentences for w in s.words][:10]
    ents = [f"{e.text}({e.type})"for e in doc.ents]
    return "|".join(pos), "|".join(ents)

    
df_sample = df.head(50).copy()
df_sample[['POS_Tages','Entities']]= df_sample['cleaned_text'].apply(lambda x:pd.Series(extract_linguistic_features(x)))
df_sample[['cleaned_text','POS_Tages','Entities']].head()



In [ ]:
df.head()

In [ ]:
df.to_csv('cleaned_news_data.csv',index=False)
print("Stage two completed successfully")
df.head()

In [ ]:
#Bag of words
bow_vectorizer = CountVectorizer(max_features=1000)
bow_matrix = bow_vectorizer.fit_transform(df['stemmed_text'].astype(str))
bow_df = pd.DataFrame(bow_matrix.toarray(),columns = bow_vectorizer.get_feature_names_out())
print("1.Bag of Words applied successfully!")
bow_df.head()


In [ ]:
#TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000,ngram_range=(1,2))
tfidf_matrix = tfidf_vectorizer.fit_transform(df['stemmed_text'].astype(str))
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),columns=tfidf_vectorizer.get_feature_names_out())
print("2.TF-IDF applied successfully!")
tfidf_df.head()


In [ ]:
#word Embedding
#Tokenize for Word2Vec 
sentences = [text.split() for text in df['stemmed_text'].astype(str)]
#Train Word2Vec
word2vec_model = Word2Vec(sentences,vector_size=100,window=5,min_count=1,workers=4)
print("3.Word2Vec Embedding created successfully!")



In [ ]:
def get_document_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

w2v_vectors = np.array([get_document_vector(tokens, word2vec_model) for tokens in sentences])
print("W2V Vectors Shape:", w2v_vectors.shape)

In [ ]:
#Save the final cleaned and processed data to a CSV file
df.to_csv('final_processed_news.csv',index=False)
print('Project Milestone:All preprocessing and vectorization stage are complete!')

In [ ]:
#Text Similarity
#Cosine similarity
#1.Calculate cosine similarity matrix based on TF-IDF
cos_sim_matrix = cosine_similarity(tfidf_matrix)
sample_score = cos_sim_matrix[0][1]
print(f"Similarity score between Arabic 0 and Article 1:{sample_score:.4f}")
print("Text Similarity calculation completed successfully!")


In [ ]:
#1.Select the most important columns for the final report
final_report = df[['cleaned_text','stemmed_text']].copy()
#2.Add the results we extracted (optional:you can add more columns)
final_report.to_csv('Final_Project_Data.csv',index=False)
print("All stage are complete. The final file 'Final_Project_Data.csv' is ready for download!")


In [ ]:
def get_live_news_v3(url):
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            title = soup.find('h1').text.strip() if soup.find('h1') else "عنوان الخبر"
            paragraphs = soup.find_all('p')
            full_text = " ".join([p.text.strip() for p in paragraphs])
            return full_text, title
        else:
            return None, "خطأ في الاتصال"
    except:
        return None, "فشل السحب"

In [ ]:

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(final_report['stemmed_text'])
y = df['label'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Done")

In [ ]:
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)

y_pred = svm_model.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(10, 7))
cm = confusion_matrix(y_test, y_pred)


sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', 
            xticklabels=svm_model.classes_, 
            yticklabels=svm_model.classes_)

plt.xlabel('Predicted Label (توقعات الموديل)')
plt.ylabel('True Label (الحقيقة)')
plt.title('Confusion Matrix: SVM Arabic News Classification')
plt.show()


In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(final_report['stemmed_text'])
y = df['label'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Done")

In [ ]:

# بناء موديل المصداقية (حقيقي/كاذب)
model_auth = SVC(kernel='linear')
model_auth.fit(X_train, y_train)

print("تم بناء نموذج المصداقية بنجاح!")

In [ ]:
# 💡 تجربة فحص خبر جديد: يمكنك تغيير النص بين القوسين واختبار الموديل
news_text = "أعلنت وزارة الصحة عن إطلاق حملة جديدة للتطعيمات في كافة المراكز"

if len(news_text) > 10:
    cleaned_data = basic_cleaning(news_text)
    tokens = tokenize_and_remove_stop(cleaned_data)
    stemmed_data = apply_stemming(tokens)
    vector_data = vectorizer.transform([stemmed_data])
    category = svm_model.predict(vector_data)[0]
    
    print(f"\n" + "="*40)
    print(f" نتيجة فحص الخبر ")
    print(f"="*40)
    print(f" النص المدخل: {news_text}")
    print(f" تصنيف الموضوع: {category}")
    print(f"="*40)
else:
    print("الرجاء إدخال نص كافٍ للتحليل.")